# Mutual Fund Analysis

This notebook analyzes mutual fund data with the following steps:
1. Drop NA rows
2. Filter for funds with beta < 1 and alpha > 1
3. Sort by beta (ascending), alpha (descending), and sharpe ratio (largest)
4. Create visualizations
5. Display filtered results

In [23]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Set default template for better looking plots
template = 'plotly_white'

In [24]:
# Read the CSV file
df = pd.read_csv('a.csv')

# Display basic information about the dataset
print("Dataset Info:")
df.info()

print("\nFirst few rows:")
df.head()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12109 entries, 0 to 12108
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    12109 non-null  object 
 1   risk    12109 non-null  object 
 2   alpha   12109 non-null  object 
 3   beta    12109 non-null  object 
 4   r2      12109 non-null  object 
 5   stddev  12109 non-null  object 
 6   sharpe  12109 non-null  object 
 7           0 non-null      float64
dtypes: float64(1), object(7)
memory usage: 756.9+ KB

First few rows:


,name,risk,alpha,beta,r2,stddev,sharpe,
0,Quant Liquid Mn IDCW-P,–,47.34,15.57,4.03,5.99,8.63,NaN
1,Quant Liquid Mn IDCW-R,–,47.34,15.57,4.03,5.99,8.63,NaN
2,Quant Liquid Dir Mn IDCW-P,–,15.06,26.05,5.33,8.7,2.51,NaN
3,Quant Liquid Dir Mn IDCW-R,–,15.06,26.05,5.33,8.7,2.51,NaN
4,DSP Credit Risk Fund Dir IDCW-P,High,14.52,-0.16,–,28.26,0.51,NaN


In [25]:
# Convert columns to numeric first, replacing '–' with NaN
df_clean = df.copy()
df_clean['beta'] = pd.to_numeric(df_clean['beta'].replace('–', 'NaN'), errors='coerce')
df_clean['alpha'] = pd.to_numeric(df_clean['alpha'].replace('–', 'NaN'), errors='coerce')
df_clean['sharpe'] = pd.to_numeric(df_clean['sharpe'].replace('–', 'NaN'), errors='coerce')

# Drop rows only where numerical columns have NA values
df_clean = df_clean.dropna(subset=['beta', 'alpha', 'sharpe'])

print(f"Original shape: {df.shape}")
print(f"Shape after cleaning: {df_clean.shape}")

Original shape: (12109, 8)
Shape after cleaning: (6883, 8)


In [26]:
# Filter rows where beta < 1 and alpha > 1
filtered_df = df_clean[
    (df_clean['beta'] < 1) & 
    (df_clean['alpha'] > 1) & 
    (df_clean['sharpe'] > df_clean['sharpe'].mean())
]

# Sort by beta (ascending), alpha (descending), and sharpe (descending)
sorted_df = filtered_df.sort_values(
    by=['beta', 'alpha', 'sharpe'],
    ascending=[True, False, False]
)

print(f"Average Sharpe ratio: {df_clean['sharpe'].mean():.2f}")
print(f"Number of funds meeting criteria: {len(filtered_df)}")
print(f"Percentage of total funds: {(len(filtered_df)/len(df_clean)*100):.1f}%")

Average Sharpe ratio: 0.23
Number of funds meeting criteria: 1223
Percentage of total funds: 17.8%


In [27]:
# Create an interactive scatter plot of beta vs alpha, with sharpe ratio as color/size
fig = px.scatter(filtered_df, x='beta', y='alpha',
                 size='sharpe', color='sharpe',
                 hover_data=['name', 'beta', 'alpha', 'sharpe'],
                 title='Fund Distribution: Beta vs Alpha',
                 labels={'beta': 'Beta (< 1)', 
                        'alpha': 'Alpha (> 1)',
                        'sharpe': 'Sharpe Ratio'},
                 template=template)

# Update layout for better visualization
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')),
                  selector=dict(mode='markers'))
fig.update_layout(coloraxis_colorbar_title='Sharpe Ratio')

fig.show()

In [ ]:
# Create an interactive bar plot of top 15 funds by Sharpe ratio
top_15 = sorted_df.head(15)

fig = px.bar(top_15, x='name', y='sharpe',
              title='Top 15 Funds by Sharpe Ratio (Beta < 1, Alpha > 1)',
              labels={'name': 'Fund Name',
                     'sharpe': 'Sharpe Ratio'},
              template=template)

# Update layout for better readability
fig.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
    height=500
)

fig.show()

In [ ]:
# Display the filtered and sorted results
print("Funds meeting criteria (beta < 1, alpha > 1), sorted by beta (asc), alpha (desc), sharpe (desc):")

# Format the display to show only relevant columns and round numbers
display_df = sorted_df[['name', 'beta', 'alpha', 'sharpe']].copy()
display_df = display_df.round({'beta': 3, 'alpha': 3, 'sharpe': 3})
display_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better visualizations
plt.style.use('seaborn')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Load the dataset
df = pd.read_csv('a.csv')
print("Original shape:", df.shape)
df.head()

In [ ]:
# Clean the data
# Drop NA rows
df_clean = df.dropna()
print("Shape after dropping NA:", df_clean.shape)

# Convert numeric columns
numeric_cols = ['beta', 'alpha', 'sharpe']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Filter and sort
df_filtered = df_clean.query('beta < 1 and alpha > 1')\
                      .sort_values(['beta', 'alpha', 'sharpe'],
                                  ascending=[True, False, False])

print("\nFinal filtered shape:", df_filtered.shape)

In [ ]:
# Visualization 1: Scatter plot of Beta vs Alpha
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_filtered, x='beta', y='alpha', size='sharpe', 
                sizes=(100, 400), alpha=0.6)
plt.title('Fund Distribution: Beta vs Alpha (size = Sharpe Ratio)')
plt.xlabel('Beta (< 1)')
plt.ylabel('Alpha (> 1)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualization 2: Top 15 funds by Sharpe ratio
top_15 = df_filtered.head(15)
plt.figure(figsize=(15, 6))
sns.barplot(data=top_15, x=top_15.index, y='sharpe')
plt.title('Top 15 Funds by Sharpe Ratio')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Fund Index')
plt.ylabel('Sharpe Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# Display filtered and sorted results
print("Showing funds with beta < 1 and alpha > 1, sorted by beta (asc), alpha (desc), and sharpe (desc)")
df_filtered